In [2]:
import csv
import os
from skimage import io
import numpy as np
from skimage.transform import resize

## Carregar o Dataset

In [3]:
CAMINHO_CSV = "annotations.csv"
PASTA_IMAGENS = "images"


imagens = []
alturas = []
pesos = []
ids = []

In [4]:
with open(CAMINHO_CSV, newline="", encoding="utf-8") as f:
    leitor = csv.reader(f)

    for linha in leitor:
        if not linha:  # ignora linhas vazias
            continue

        id_img = linha[0].strip()
        altura = float(linha[1])
        peso = float(linha[2])

        caminho_img = os.path.join(PASTA_IMAGENS, f"{id_img}.jpg")

        if not os.path.exists(caminho_img):
            print(f"Imagem não encontrada: {caminho_img}")
            continue

        img = io.imread(caminho_img)

        ids.append(id_img)
        imagens.append(img)
        alturas.append(altura)
        pesos.append(peso)

print(f"{len(imagens)} imagens carregadas")
print("Shape da primeira imagem:", imagens[0].shape)
print("Primeiro registro:", ids[0], alturas[0], pesos[0])

513 imagens carregadas
Shape da primeira imagem: (256, 256, 3)
Primeiro registro: 001 123.0 279.0


## Normalização e Divisão Treinamento

### Redimensionalizar e Normalizar

- Redimensionaliza para 128x128 e normaliza pixeis para [0,1], em vez de [0, 255].

In [18]:
tamanho_imagem = (128, 128)  # Defina o tamanho desejado para as imagens

imagens_redimensionadas = []
for img in imagens:
    img = resize(img, tamanho_imagem, anti_aliasing=True)
    imagens_redimensionadas.append(img)
    
X = np.array(imagens_redimensionadas, dtype=np.float32)

In [17]:
X.shape

(513, 128, 128, 3)

In [ ]:
# Normalizar com X.clip
X = X.clip(0.0, 1.0)  # Normaliza os valores para o intervalo [0, 1]
X.min(), X.mean() , X.max()

"""
logica do x_clip

import time
t0 = time.time()

iterations = 0
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        for k in range(X.shape[2]):
            for c in range(X.shape[3]):
                iterations += 1
                if X[i, j, k, c] < 0.0:
                    X[i, j, k, c] = 0.0
                elif X[i, j, k, c] > 1.0:
                    X[i, j, k, c] = 1.0
                print(f"Iteração {iterations}")
t_loop = time.time() - t0
"""

(0.0, 0.40776643, 1.0)

### Carregar vetores de treinamento/validação

In [31]:
y = np.array(list(zip(alturas, pesos)), dtype=np.float32)

X_train, y_train = X[50:], y[50:]
X_val, y_val = X[:50], y[:50]

mu = y_train.mean(axis=0)
std = y_train.std(axis=0)
y_train = (y_train - mu) / std

y_train.mean(), y_train.std()


(4.1195435e-08, 1.0000001)